# 1. LangChain 개요 & API 연동
**(Deep Agents 체험 → LangChain 내부 구조 이해)**

**LangChain v1 프레임워크**를 활용한 LLM 기반 에이전트 개발 입문:
- **Deep Agents 첫 체험**: `create_deep_agent()`로 동작하는 에이전트를 먼저 경험
- **하네스(Harness) 개념**: 도구 + 메모리 + 컨텍스트 = 에이전트 인프라
- **OpenAI API 연동**: `ChatOpenAI`로 LLM 호출 — `.env` 기반 키 관리
- **LangChain 핵심 구조**: 3-Layer 아키텍처 — Model, Tools, Middleware
- **ChatModel & Messages**: `invoke()` / `stream()` / `batch()` 호출 패턴
- **Tool & Memory**: `@tool` 데코레이터, `InMemorySaver` 기반 멀티턴 대화
- **Middleware**: 에이전트 파이프라인에 훅을 삽입하는 미들웨어 시스템
- **첫 번째 Agent**: `create_agent()`로 도구 + 메모리를 갖춘 에이전트 생성

---

## 0) 환경 설정 — pip install, .env, API key

In [4]:
# [1-1] : 라이브러리 설치
# [핵심] 이 노트북에서 사용하는 모든 패키지 — 첫 실행 시 한 번만 필요
# [주의] 버전 충돌 시 런타임 재시작 후 재실행
!uv pip install -q langchain langchain-core langchain-openai langgraph python-dotenv langfuse deepagents

In [5]:
# [1-2] : 환경변수 로드 및 API 키 검증
# [핵심] .env 파일에서 API 키를 로드 — 하드코딩 금지 원칙
# [주의] override=True → 셀 재실행 시 .env 값으로 덮어씀

import os
from dotenv import load_dotenv

load_dotenv(override=True)  # .env 파일에서 환경변수 로드

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # API 키 가져오기
assert OPENAI_API_KEY, "OPENAI_API_KEY가 .env에 설정되어 있지 않습니다."  # 키 존재 확인
print("API 키 로드 완료")

API 키 로드 완료


---

## Deep Agents로 Agent 첫 체험

LangChain의 내부 구조를 배우기 전에, **완성된 에이전트가 어떻게 동작하는지** 먼저 체험해 본다.  
`create_deep_agent()`는 LangChain 팀이 만든 **에이전트 하네스(Agent Harness)** 프레임워크인 Deep Agents의 핵심 함수이다.

- 도구, 메모리, 컨텍스트 관리를 **자동으로 조립**하여 바로 사용할 수 있는 에이전트를 생성한다
- 내부적으로 **LangChain** 컴포넌트와 **LangGraph** 실행 엔진을 사용한다
- 반환되는 객체는 LangGraph의 `CompiledStateGraph` — `invoke()`, `stream()` 등 동일한 인터페이스를 사용한다

In [6]:
# [1-0] : Deep Agents로 Agent 첫 체험
# [핵심] create_deep_agent는 도구+메모리+컨텍스트를 자동으로 조립하는 하네스
# [주의] deepagents 패키지가 설치되어 있어야 함 (pip install 셀에서 설치됨)

from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI

agent = create_deep_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[],
    system_prompt="당신은 친절한 AI 어시스턴트입니다.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요! AI 에이전트가 무엇인지 간단히 설명해주세요."}]}
)
print(result["messages"][-1].content)

AI 에이전트는 인공지능 기술을 활용하여 특정 작업이나 문제를 자동으로 수행하는 소프트웨어 또는 시스템입니다. 주어진 목표를 달성하기 위해 데이터를 수집, 분석, 판단, 행동하는 능력을 가지며, 사용자의 요청에 응답하거나 환경과 상호작용할 수 있습니다.


### 하네스(Harness) 개념

**하네스(Harness)**란 에이전트가 동작하기 위해 필요한 **인프라 전체**를 의미한다:

| 구성 요소 | 역할 | Deep Agents에서의 구현 |
|-----------|------|------------------------|
| **도구(Tools)** | 에이전트가 외부와 상호작용하는 수단 | `tools` 파라미터, 빌트인 파일 도구 |
| **메모리(Memory)** | 대화 이력 및 장기 기억 관리 | `checkpointer`, `StoreBackend` |
| **컨텍스트(Context)** | 토큰 한도 내 효율적 정보 관리 | 자동 오프로딩, 요약 미들웨어 |

> **하네스 = 도구 + 메모리 + 컨텍스트**  
> `create_deep_agent()`는 이 세 가지를 자동으로 조립하여 바로 사용 가능한 에이전트를 만들어 준다.  
> 이 교육 과정에서는 이 하네스의 **내부를 직접 만들어보면서** 에이전트의 동작 원리를 이해한다.

In [7]:
# [1-0b] : 커스텀 도구를 가진 Deep Agent 체험
# [핵심] tools에 Python 함수를 전달하면 에이전트가 자율적으로 사용
# [주의] docstring과 타입 힌트가 에이전트에게 도구 설명으로 전달됨

def greet(name: str) -> str:
    """사용자의 이름을 받아 환영 인사를 생성합니다."""
    return f"안녕하세요, {name}님! SK하이닉스 AI 에이전트 교육에 오신 것을 환영합니다."

tool_agent = create_deep_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[greet],
    system_prompt="사용자가 이름을 알려주면 반드시 greet 도구를 사용하여 인사하세요.",
)

result = tool_agent.invoke(
    {"messages": [{"role": "user", "content": "제 이름은 홍길동입니다."}]}
)
print(result["messages"][-1].content)

안녕하세요, 홍길동님! 무엇을 도와드릴까요?


---

## 1) OpenAI API 연동 — ChatOpenAI 초기화 및 기본 호출

**`ChatOpenAI`** 는 OpenAI 호환 LLM을 래핑하는 LangChain 클래스이다.
- `.env`에 설정된 `OPENAI_API_KEY`를 자동으로 읽어온다
- `model` 파라미터로 사용할 모델을 지정한다 (예: `"gpt-4.1"`)
- `temperature` 파라미터로 응답의 무작위성을 제어한다 — 0에 가까울수록 결정적

In [8]:
# [1-4] : ChatOpenAI 모델 초기화
# [핵심] LangChain에서 OpenAI 모델을 사용하기 위한 기본 진입점
# [주의] model 파라미터는 OpenAI에서 지원하는 모델명이어야 함

from langchain_openai import ChatOpenAI  # OpenAI 전용 Chat 모델 래퍼

model = ChatOpenAI(
    model="gpt-4.1",       # 사용할 모델 지정
    temperature=0,          # 0 → 결정적 출력 (재현성 확보)
)
print("모델 초기화 완료:", model.model_name)

모델 초기화 완료: gpt-4.1


In [9]:
# [1-5] : 모델 기본 호출 (문자열 입력)
# [핵심] invoke()에 문자열을 전달하면 HumanMessage로 자동 변환
# [주의] 응답은 AIMessage 객체 — .content 속성으로 텍스트 추출

response = model.invoke("안녕하세요! 한 문장으로 자기소개 해주세요.")

print("응답 내용:", response.content) # 텍스트 내용

응답 내용: 안녕하세요! 저는 여러분의 질문에 친절하고 정확하게 답변해드리는 인공지능 챗봇입니다.


### `init_chat_model()` — 프로바이더 통합 초기화

LangChain v1은 `init_chat_model()` 함수로 다양한 프로바이더의 모델을 **문자열 하나**로 초기화할 수 있다.

| 프로바이더 | 모델 문자열 | 필요 패키지 |
|-----------|-----------|------------|
| OpenAI | `"openai:gpt-4.1"` | `langchain-openai` |
| Anthropic | `"anthropic:claude-sonnet-4-6"` | `langchain-anthropic` |
| Google | `"google:gemini-2.0-flash"` | `langchain-google-genai` |
| Ollama | `"ollama:llama3"` | `langchain-ollama` |

In [10]:
# [1-6] : init_chat_model()로 통합 초기화
# [핵심] 프로바이더:모델명 형식 → 프로바이더별 패키지가 설치되어 있으면 자동 연결
# [주의] 해당 프로바이더 패키지가 설치되어 있어야 동작함

from langchain.chat_models import init_chat_model  # 통합 모델 초기화 함수

model_via_init = init_chat_model(
    model="openai:gpt-4.1",  # 프로바이더:모델명 형식
    temperature=0,
)

response = model_via_init.invoke("2 + 3은 얼마인가요?")
print("init_chat_model 응답:", response.content)

init_chat_model 응답: 2 + 3은 5입니다.


---

## 2) LangChain 핵심 구조 이해 — 3-Layer Architecture

> **Deep Agents 내부에서 LangChain이 동작합니다. 이제 그 내부를 직접 만들어봅시다.**

LangChain v1은 **3가지 레이어**로 구성된 에이전트 프레임워크이다.

| 레이어 | 역할 | 핵심 API |
|--------|------|----------|
| **Deep Agents** | 사전 구축된 에이전트 (코딩, 리서치 등) | `create_deep_agent()` — 빠른 프로토타이핑 |
| **LangGraph** | 복잡한 워크플로 — 상태 그래프, 체크포인터 | `StateGraph`, `InMemorySaver` |
| **LangChain** | 에이전트 생성의 핵심 — 모델, 도구, 미들웨어 | `create_agent()`, `@tool`, `ChatOpenAI` |

```
┌─────────────────────────────────────┐
│         Deep Agents (하네스)          │  ← create_deep_agent()
│  도구 + 메모리 + 컨텍스트 자동 조립    │
├─────────────────────────────────────┤
│         LangChain (빌딩 블록)         │  ← create_agent(), @tool, ChatOpenAI
│  모델, 도구, 미들웨어 정의            │
├─────────────────────────────────────┤
│         LangGraph (실행 엔진)         │  ← StateGraph, CompiledStateGraph
│  상태 그래프, 체크포인터, 스트리밍     │
└─────────────────────────────────────┘
```

### 핵심 컴포넌트

| 컴포넌트 | 설명 | 주요 API |
|----------|------|----------|
| **Model** | LLM — 에이전트의 "두뇌" 역할 | `ChatOpenAI`, `init_chat_model()` |
| **Tools** | 에이전트가 사용하는 함수 — 검색, 계산, API 호출 | `@tool` 데코레이터 |
| **Agent** | 모델 + 도구를 결합한 실행 단위 | `create_agent()` |
| **Memory** | 대화 히스토리 저장 및 관리 | `InMemorySaver` |
| **Middleware** | 요청/응답 파이프라인에 로직 삽입 | `@before_model`, `@after_model` |

### ReAct 패턴 — 에이전트의 기본 동작 방식

LangChain v1 에이전트는 **ReAct (Reasoning + Acting)** 패턴으로 동작한다:

```
사용자 질문 → [추론(Reasoning)] → [행동(Action: 도구 호출)] → [관찰(Observation)] → 최종 응답
```

- 에이전트가 도구 사용 여부를 **자율적으로 판단**한다
- 복잡한 작업을 **여러 단계로 분해**하여 처리한다
- `create_agent()`로 생성된 에이전트는 내부적으로 **LangGraph 그래프**로 실행된다

In [11]:
# [1-7] : LangChain v1 핵심 import 확인
# [핵심] 이 네 가지가 LangChain v1의 핵심 빌딩 블록
# [주의] create_agent는 v1 전용 — v0.x의 create_agent와 다름

from langchain.agents import create_agent       # 에이전트 생성 함수
from langchain.tools import tool                 # 도구 데코레이터
from langchain_openai import ChatOpenAI           # OpenAI Chat 모델
from langgraph.checkpoint.memory import InMemorySaver  # 메모리 체크포인터

print("핵심 모듈 임포트 완료")
print("  - create_agent: 에이전트 생성 (v1)")
print("  - @tool: 도구 데코레이터")
print("  - ChatOpenAI: OpenAI 모델 래퍼")
print("  - InMemorySaver: 메모리 체크포인터")

핵심 모듈 임포트 완료
  - create_agent: 에이전트 생성 (v1)
  - @tool: 도구 데코레이터
  - ChatOpenAI: OpenAI 모델 래퍼
  - InMemorySaver: 메모리 체크포인터


---

## 3) ChatModel 기본 사용법 — invoke / stream / batch

LangChain의 모든 ChatModel은 **세 가지 호출 패턴**을 지원한다:

| 메서드 | 설명 | 반환 타입 |
|--------|------|----------|
| `invoke()` | 단일 입력 → 단일 응답 | `AIMessage` |
| `stream()` | 토큰 단위 실시간 스트리밍 | `Iterator[AIMessageChunk]` |
| `batch()` | 여러 입력을 동시 처리 | `List[AIMessage]` |



In [12]:
# [1-8] : invoke() — 단일 호출
# [핵심] 가장 기본적인 호출 방식 — 전체 응답을 한 번에 반환
# [주의] 긴 응답은 완료까지 대기해야 함 → 대화형 UI에는 stream() 권장

response = model.invoke(
    "LangChain이 무엇인지 한 문장으로 설명해 주세요."  # 문자열 입력
)
print("invoke 결과:", response.content)

invoke 결과: LangChain은 다양한 언어 모델과 데이터 소스를 연결하여 복잡한 자연어 처리 애플리케이션을 쉽게 개발할 수 있도록 도와주는 오픈소스 프레임워크입니다.


In [13]:
# [1-9] : stream() — 토큰 단위 스트리밍
# [핵심] 토큰이 생성되는 대로 실시간 출력 — 사용자 체감 속도 향상
# [주의] 각 chunk는 AIMessageChunk 객체 — .content로 텍스트 추출

print("stream 결과: ", end="")
for chunk in model.stream(
    "1부터 5까지 세어 주세요."  # 스트리밍 입력
):
    print(chunk.content, end="", flush=True)  # 토큰 단위 출력
print()  # 줄바꿈

stream 결과: 네, 1부터 5까지 세어드리겠습니다.

1  
2  
3  
4  
5


In [14]:
# [1-10] : batch() — 여러 입력 동시 처리
# [핵심] 여러 요청을 병렬로 처리 — 개별 invoke() 반복보다 효율적
# [주의] 입력은 리스트 형태 — 각 요소가 독립적인 요청

responses = model.batch(
    [
        "Python이란?",           # 요청 1
        "JavaScript란?",         # 요청 2
        "LangChain이란?",        # 요청 3
    ]
)

for i, resp in enumerate(responses, 1):
    print(f"응답 {i}: {resp.content[:80]}...")  # 앞 80자만 출력
    print()

응답 1: Python이란?

Python은 1991년 네덜란드의 귀도 반 로섬(Guido van Rossum)이 개발한 고급 프로그래밍 언어입니다. 문법...

응답 2: JavaScript란?

JavaScript는 웹 페이지를 동적으로 만들기 위해 주로 사용되는 **프로그래밍 언어**입니다. 1995년에 넷스케...

응답 3: LangChain은 자연어 처리(NLP)와 인공지능(AI) 분야에서 사용되는 오픈소스 프레임워크로, 대형 언어 모델(LLM, Large Lang...



---

## 4) Messages 타입 이해 — System / Human / AI / Tool

LLM은 **메시지 리스트**를 입력으로 받는다. 각 메시지에는 **역할(role)**이 있다.

| 메시지 타입 | 역할 | 설명 |
|------------|------|------|
| `SystemMessage` | 시스템 | 모델의 행동 지침 — 페르소나, 톤, 규칙 설정 |
| `HumanMessage` | 사용자 | 사용자 입력 — 텍스트, 이미지 등 멀티모달 지원 |
| `AIMessage` | AI | 모델 응답 — `tool_calls`, `usage_metadata` 포함 |
| `ToolMessage` | 도구 | 도구 실행 결과를 모델에 전달 |

LangChain은 메시지 입력을 **세 가지 형식**으로 지원한다:
1. **문자열**: `model.invoke("Hello")` → `HumanMessage`로 자동 변환
2. **메시지 객체 리스트**: `[SystemMessage(...), HumanMessage(...)]`
3. **딕셔너리 리스트**: `[{"role": "system", "content": "..."}]` — OpenAI API 호환



In [15]:
# [1-11] : 메시지 객체를 사용한 대화 구성
# [핵심] SystemMessage로 모델의 페르소나를 설정 → 같은 질문에 다른 응답
# [주의] SystemMessage는 메시지 리스트의 맨 앞에 배치해야 함

from langchain.messages import SystemMessage, HumanMessage, AIMessage

# 과학자 페르소나
messages_scientist = [
    SystemMessage(content="당신은 물리학자입니다. 정확하게 답하세요."),
    HumanMessage(content="중력에 대해 설명해 주세요."),
]

r1 = model.invoke(messages_scientist)
print("[과학자]", r1.content[:150])

[과학자] 중력(gravity)은 질량을 가진 모든 물체 사이에 작용하는 자연의 네 가지 기본 힘 중 하나입니다. 중력은 물체가 서로를 끌어당기는 힘으로, 우리가 지구에 붙어 있을 수 있게 하고, 달이 지구 주위를 돌게 하며, 태양계와 우주의 구조를 결정짓는 힘입니다.

**뉴턴


In [16]:
# 어린이 교사 페르소나
messages_teacher = [
    SystemMessage(content="5살 아이에게 설명하듯이 쉬운 단어를 사용하세요."),
    HumanMessage(content="중력에 대해 설명해 주세요."),
]

r2 = model.invoke(messages_teacher)
print("[교사]", r2.content[:150])

[교사] 중력은 우리를 땅에 붙잡아 주는 힘이에요. 예를 들어, 우리가 점프하면 다시 땅으로 떨어지죠? 그게 바로 중력 때문이에요. 중력은 지구가 우리를 끌어당기는 힘이에요. 그래서 우리가 하늘로 날아가지 않고 땅 위에 있을 수 있는 거예요. 공도 던지면 땅으로 떨어지고, 사과


In [17]:
# [1-12] : 딕셔너리 형식 + 대화 이력 관리
# [핵심] AIMessage를 이력에 포함시키면 모델이 이전 대화 맥락을 유지
# [주의] 딕셔너리 형식은 OpenAI API와 동일 — 마이그레이션에 유리

conversation = [
    {"role": "system", "content": "당신은 수학 튜터입니다. 간결하게 답하세요."},
    {"role": "user", "content": "소수란 무엇인가요?"},
    {"role": "assistant", "content": "소수는 1과 자기 자신만으로 나누어 떨어지는 1보다 큰 자연수입니다."},
    {"role": "user", "content": "5가지 예시를 알려주세요."},  # 이전 맥락 참조
]

response = model.invoke(conversation)
print("대화 이력 응답:", response.content)

대화 이력 응답: 2, 3, 5, 7, 11


In [18]:
# [1-13] : AIMessage 응답 객체 상세 분석
# [핵심] AIMessage에는 텍스트 외에 usage_metadata(토큰 사용량) 등이 포함
# [주의] response_metadata는 프로바이더마다 구조가 다를 수 있음

response = model.invoke("안녕하세요!")

print("타입:", type(response).__name__)               # AIMessage
print("내용:", response.content)                        # 텍스트 응답
print("토큰 사용량:", response.usage_metadata)           # 토큰 정보
print("도구 호출:", response.tool_calls)                 # 도구 호출 (없으면 빈 리스트)

타입: AIMessage
내용: 안녕하세요! 무엇을 도와드릴까요? 😊
토큰 사용량: {'input_tokens': 10, 'output_tokens': 12, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
도구 호출: []


---

## 5) Tool 개념 및 활용 — @tool 데코레이터, Tool Calling

**Tool(도구)**은 에이전트가 호출할 수 있는 함수이다.

- `@tool` 데코레이터를 붙이면 함수가 에이전트용 도구로 변환된다
- LangChain은 함수의 **이름**, **docstring**, **타입 힌트**를 자동 파싱하여 스키마를 생성한다
- 에이전트는 이 스키마를 보고 **언제, 어떤 인자로** 도구를 호출할지 자율적으로 판단한다

도구 정의 시 필수 사항:
- **docstring**: 에이전트가 도구의 용도를 이해하는 데 사용
- **타입 힌트**: 에이전트가 올바른 인자를 전달하는 데 사용



In [19]:
# [1-14] : @tool 데코레이터로 커스텀 도구 정의
# [핵심] docstring이 에이전트에게 도구 설명으로 전달됨 — 반드시 작성
# [주의] 타입 힌트 누락 시 에이전트가 잘못된 인자를 전달할 수 있음

from langchain.tools import tool  # 도구 데코레이터

@tool
def add(a: int, b: int) -> int:
    """두 수를 더합니다."""
    return a + b  # 단순 덧셈

@tool
def multiply(a: int, b: int) -> int:
    """두 수를 곱합니다."""
    return a * b  # 단순 곱셈

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다."""
    weather_data = {"서울": "맑음, 15\u00b0C", "도쿄": "흐림, 12\u00b0C", "뉴욕": "비, 8\u00b0C"}
    return weather_data.get(city, f"{city}의 날씨 데이터가 없습니다.")  # 딕셔너리 조회

# 도구 스키마 확인
print("도구 목록:")
for t in [add, multiply, get_weather]:
    print(f"  - {t.name}: {t.description}")

print("\nadd 입력 스키마:", add.args_schema.model_json_schema())

도구 목록:
  - add: 두 수를 더합니다.
  - multiply: 두 수를 곱합니다.
  - get_weather: 도시의 현재 날씨를 조회합니다.

add 입력 스키마: {'description': '두 수를 더합니다.', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'add', 'type': 'object'}


In [20]:
# [1-15] : 도구를 에이전트에 연결하여 실행
# [핵심] create_agent()에 tools 리스트를 전달 → 에이전트가 자동으로 도구 선택
# [참고] 앞서 create_deep_agent로 만든 Agent가 내부적으로 create_agent를 사용합니다
# [주의] v1에서는 create_agent() 대신 반드시 create_agent() 사용

from langchain.agents import create_agent  # v1 에이전트 생성 함수

tool_agent = create_agent(
    model=model,                   # 사용할 LLM
    tools=[add, multiply],         # 사용 가능한 도구 리스트
    system_prompt="당신은 수학 도우미입니다. 반드시 제공된 도구를 사용하여 계산하세요.",
)

result = tool_agent.invoke(
    {"messages": [{"role": "user", "content": "15 + 27은 얼마인가요?"}]}
)

# 최종 응답 출력
print("에이전트 응답:", result["messages"][-1].content)

에이전트 응답: 15 + 27은 42입니다.


In [21]:
# [1-16] : 에이전트 메시지 흐름 확인 (ReAct 루프 관찰)
# [핵심] 에이전트는 user → ai(tool_call) → tool(결과) → ai(응답) 순서로 동작
# [주의] ai 메시지의 tool_calls 속성에 도구 호출 정보가 담겨 있음

result = tool_agent.invoke(
    {"messages": [{"role": "user", "content": "12 * 8을 계산한 다음 결과에 5를 더해 주세요."}]}
)

print("메시지 흐름:")
print("=" * 50)
for msg in result["messages"]:
    role = msg.type if hasattr(msg, "type") else "unknown"  # 메시지 역할
    content = msg.content if hasattr(msg, "content") else ""  # 메시지 내용
    tool_calls = getattr(msg, "tool_calls", None)  # 도구 호출 정보
    print(f"[{role}] {content[:200]}")
    if tool_calls:
        print(f"       도구 호출: {[tc['name'] for tc in tool_calls]}")
    print("-" * 50)

메시지 흐름:
[human] 12 * 8을 계산한 다음 결과에 5를 더해 주세요.
--------------------------------------------------
[ai] 
       도구 호출: ['multiply']
--------------------------------------------------
[tool] 96
--------------------------------------------------
[ai] 
       도구 호출: ['add']
--------------------------------------------------
[tool] 101
--------------------------------------------------
[ai] 12 × 8 = 96이고, 여기에 5를 더하면 101입니다. 

최종 답: 101
--------------------------------------------------


---

## 6) Memory 개념 및 활용 — InMemorySaver, thread_id, 멀티턴 대화

**Memory(메모리)**는 에이전트가 이전 대화를 기억하는 메커니즘이다.

- **`InMemorySaver`**: 메모리 내 체크포인터 — 에이전트 상태를 저장
- **`thread_id`**: 대화 세션 식별자 — 같은 ID면 이전 대화 컨텍스트 유지
- 서로 다른 `thread_id`는 **완전히 독립된 대화** 세션

| 구분 | 단기 메모리 (Checkpointer) | 장기 메모리 (Store) |
|------|--------------------------|--------------------|  
| 범위 | 하나의 `thread_id` 내 | 모든 세션에 걸쳐 |
| 저장 대상 | 대화 메시지 히스토리 | 사용자 선호도, 학습 데이터 |
| 수명 | 세션 단위 | 명시적 삭제 전까지 영구 |



In [22]:
# [1-17] : InMemorySaver로 멀티턴 대화 구현
# [핵심] checkpointer에 InMemorySaver를 전달하면 thread_id별 대화 상태 저장
# [주의] thread_id는 configurable 딕셔너리 안에 넣어야 함

from langgraph.checkpoint.memory import InMemorySaver  # 메모리 체크포인터

checkpointer = InMemorySaver()  # 체크포인터 인스턴스 생성

memory_agent = create_agent(
    model=model,
    tools=[add, multiply],
    system_prompt="당신은 수학 도우미입니다. 대화 맥락을 기억하세요.",
    checkpointer=checkpointer,  # 메모리 체크포인터 연결
)

# thread_id로 대화 세션을 식별
config_session = {"configurable": {"thread_id": "math-session-1"}}

In [23]:
# 첫 번째 대화
result1 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "7 * 6은 얼마인가요?"}]},
    config={**config_session},
)
print("첫 번째 응답:", result1["messages"][-1].content)

첫 번째 응답: 7 × 6은 42입니다.


In [24]:
# 두 번째 대화 — 이전 결과를 기억하는지 확인
result2 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "이전 결과에 10을 더해 주세요."}]},
    config={**config_session},  # 같은 thread_id 사용
)
print("두 번째 응답:", result2["messages"][-1].content)

두 번째 응답: 이전 결과인 42에 10을 더하면 52가 됩니다.


In [25]:
# [1-18] : 다른 thread_id → 독립된 대화 세션
# [핵심] thread_id가 다르면 이전 대화 컨텍스트가 공유되지 않음
# [주의] 사용자별/세션별로 고유한 thread_id를 부여해야 함

config_new_session = {"configurable": {"thread_id": "math-session-2"}}  # 새 세션

result3 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "이전 결과가 무엇이었나요?"}]},
    config={**config_new_session},
)
print("새 세션 응답:", result3["messages"][-1].content)
print("→ 다른 thread_id이므로 이전 대화를 알 수 없음")

새 세션 응답: 아직 이전에 계산한 결과가 없습니다. 새로운 계산이나 질문이 있으시면 말씀해 주세요!
→ 다른 thread_id이므로 이전 대화를 알 수 없음


---

## 7) Middleware 개념 및 활용 — 에이전트 파이프라인 훅

**Middleware(미들웨어)**는 에이전트 실행 파이프라인의 각 단계에 **훅(hook)**을 추가하여 동작을 제어하는 메커니즘이다.

| 훅 | 데코레이터 | 실행 시점 | 주요 용도 |
|-----|-----------|----------|----------|
| Before Model | `@before_model` | 모델 호출 전 | 입력 검증, 가드레일 |
| After Model | `@after_model` | 모델 응답 후 | 출력 로깅, 필터링 |
| Wrap Model | `@wrap_model_call` | 모델 호출 감싸기 | 재시도, 폴백, 캐싱 |
| Wrap Tool | `@wrap_tool_call` | 도구 호출 감싸기 | 타이밍, 에러 핸들링 |
| Dynamic Prompt | `@dynamic_prompt` | 프롬프트 생성 시 | 런타임 프롬프트 변경 |

```
사용자 입력 → [@before_model] → [모델 호출] → [@after_model] → 도구 호출 → ... → 최종 응답
```

In [26]:
# [1-19] : @before_model — 모델 호출 전 로깅 미들웨어
# [핵심] 모델에 전달되는 메시지 수를 로깅 — 디버깅 및 모니터링에 유용
# [주의] state["messages"]에 현재까지의 전체 대화 이력이 담겨 있음

from langchain.agents.middleware import before_model  # before_model 데코레이터

@before_model
def log_model_input(state, runtime):
    """모델 호출 전 메시지 수를 로깅합니다."""
    msg_count = len(state["messages"])  # 현재 메시지 수
    print(f"  [before_model] 메시지 {msg_count}개 전달")

In [27]:
# [1-20] : @after_model — 모델 응답 후 로깅 미들웨어
# [핵심] 모델 출력과 도구 호출 여부를 로깅 — 에이전트 동작 추적에 유용
# [주의] after_model은 모델 응답이 생성된 후 매번 호출됨

from langchain.agents.middleware import after_model  # after_model 데코레이터

@after_model
def log_model_output(state, runtime):
    """모델 응답 후 출력을 로깅합니다."""
    msg = state["messages"][-1] if state["messages"] else None
    if msg and hasattr(msg, "content") and msg.content:
        print(f"  [after_model] 응답: {msg.content[:80]}...")  # 앞 80자
    if msg and hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"  [after_model] 도구 호출: {[tc['name'] for tc in msg.tool_calls]}")

In [28]:
# [1-21] : 미들웨어를 적용한 에이전트 실행
# [핵심] middleware 파라미터에 리스트로 전달 → 순서대로 적용
# [주의] 여러 미들웨어를 조합하면 실행 순서에 주의

middleware_agent = create_agent(
    model=model,
    tools=[add, multiply],
    system_prompt="당신은 수학 도우미입니다.",
    middleware=[log_model_input, log_model_output],  # 미들웨어 리스트
)

In [29]:
print("미들웨어 에이전트 실행:")
print("=" * 50)
result = middleware_agent.invoke(
    {"messages": [{"role": "user", "content": "5 + 3을 계산해 주세요."}]}
)
print("=" * 50)
print("최종 응답:", result["messages"][-1].content)

미들웨어 에이전트 실행:
  [before_model] 메시지 1개 전달
  [after_model] 도구 호출: ['add']
  [before_model] 메시지 3개 전달
  [after_model] 응답: 5 + 3 = 8입니다....
최종 응답: 5 + 3 = 8입니다.


In [30]:
# [1-22] : @dynamic_prompt — 런타임 프롬프트 동적 변경
# [핵심] 실행 시점의 날짜/시간 등 동적 정보를 시스템 프롬프트에 주입
# [주의] dynamic_prompt는 system_prompt 대신 사용 — 둘을 동시에 쓰지 않음

from langchain.agents.middleware import dynamic_prompt  # dynamic_prompt 데코레이터
from datetime import datetime

@dynamic_prompt
def add_datetime_context(request):
    """시스템 프롬프트에 현재 날짜/시간을 추가합니다."""
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # 현재 시각
    return f"현재 날짜와 시간: {now}\n\n당신은 유용한 어시스턴트입니다."

In [31]:
dynamic_agent = create_agent(
    model=model,
    tools=[],
    middleware=[add_datetime_context],  # dynamic_prompt 미들웨어 전달
)

result = dynamic_agent.invoke(
    {"messages": [{"role": "user", "content": "현재 날짜와 시간이 어떻게 되나요?"}]}
)
print("동적 프롬프트 응답:", result["messages"][-1].content)

동적 프롬프트 응답: 현재 날짜와 시간은 2026년 6월 15일 08시 21분 13초입니다.


---

## 8) 첫 번째 Agent 만들기 — create_agent() with Tools + Memory

지금까지 배운 **Tool**, **Memory**, **Middleware**를 모두 조합하여 실용적인 에이전트를 구축한다.

구성 요소:
- **Model**: `ChatOpenAI` — 에이전트의 두뇌
- **Tools**: 계산 도구 + 정보 검색 도구
- **Memory**: `InMemorySaver` — 멀티턴 대화 지원
- **Middleware**: 입출력 로깅
- **Streaming**: `stream_mode="updates"` — 실시간 진행 상황 확인



In [32]:
# [1-23] : 종합 에이전트 생성 — Tool + Memory + Middleware 결합
# [핵심] create_agent()의 모든 주요 파라미터를 조합한 완성형 에이전트
# [주의] checkpointer가 있으면 반드시 config에 thread_id를 포함해야 함

from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

# --- 도구 정의 ---
@tool
def calculate(a: int, b: int, op: str) -> str:
    """두 수에 대해 사칙연산을 수행합니다. op는 'add', 'sub', 'mul', 'div' 중 하나입니다."""
    ops = {
        "add": a + b,
        "sub": a - b,
        "mul": a * b,
        "div": a / b if b != 0 else "0으로 나눌 수 없습니다",
    }
    result = ops.get(op, "지원하지 않는 연산입니다")
    return f"{a} {op} {b} = {result}"

@tool
def search_info(query: str) -> str:
    """주어진 주제에 대한 정보를 검색합니다."""
    # 시뮬레이션된 검색 결과
    info = {
        "LangChain": "LangChain은 LLM 기반 에이전트를 구축하기 위한 오픈소스 프레임워크입니다.",
        "Python": "Python은 범용 프로그래밍 언어로, AI/ML 분야에서 널리 사용됩니다.",
        "SK하이닉스": "SK하이닉스는 세계적인 반도체 메모리 기업입니다.",
    }
    # query에 포함된 키워드로 검색
    for key, value in info.items():
        if key.lower() in query.lower():
            return value
    return f"'{query}'에 대한 정보를 찾을 수 없습니다."

# --- 에이전트 생성 ---
full_agent = create_agent(
    model=model,                                     # LLM 모델
    tools=[calculate, search_info],                  # 도구 리스트
    system_prompt="당신은 계산과 정보 검색이 가능한 AI 어시스턴트입니다. "
                  "도구를 적극적으로 활용하세요.",      # 시스템 프롬프트
    checkpointer=InMemorySaver(),                     # 메모리 체크포인터
    middleware=[log_model_input, log_model_output],    # 미들웨어
)

print("종합 에이전트 생성 완료")
print(f"  타입: {type(full_agent).__name__}")
print(f"  도구: {[t.name for t in [calculate, search_info]]}")

종합 에이전트 생성 완료
  타입: CompiledStateGraph
  도구: ['calculate', 'search_info']


In [33]:
# [1-24] : 에이전트 실행 — invoke로 기본 호출
# [핵심] 에이전트가 질문을 분석하고 적절한 도구를 자율적으로 선택
# [주의] config에 thread_id를 반드시 포함 — 메모리 체크포인터 필수 요구사항

config_full = {"configurable": {"thread_id": "full-agent-1"}}

print("=== 첫 번째 질문: 계산 ===")
result1 = full_agent.invoke(
    {"messages": [{"role": "user", "content": "15와 27을 더해 주세요."}]},
    config={**config_full},
)
print("응답:", result1["messages"][-1].content)
print()

print("=== 두 번째 질문: 이전 대화 참조 ===")
result2 = full_agent.invoke(
    {"messages": [{"role": "user", "content": "방금 결과에 100을 더하면 얼마인가요?"}]},
    config={**config_full},  # 같은 thread_id → 대화 기억
)
print("응답:", result2["messages"][-1].content)

=== 첫 번째 질문: 계산 ===
  [before_model] 메시지 1개 전달
  [after_model] 도구 호출: ['calculate']
  [before_model] 메시지 3개 전달
  [after_model] 응답: 15와 27을 더하면 42입니다....
응답: 15와 27을 더하면 42입니다.

=== 두 번째 질문: 이전 대화 참조 ===
  [before_model] 메시지 5개 전달
  [after_model] 도구 호출: ['calculate']
  [before_model] 메시지 7개 전달
  [after_model] 응답: 방금 결과인 42에 100을 더하면 142입니다....
응답: 방금 결과인 42에 100을 더하면 142입니다.


In [34]:
# [1-25] : 에이전트 스트리밍 — stream_mode="updates"로 진행 상황 실시간 확인
# [핵심] 각 노드(model, tools)의 업데이트를 실시간으로 관찰 가능
# [주의] stream_mode="updates"는 노드별 업데이트만 반환 — 전체 상태는 "values" 사용

config_stream = {"configurable": {"thread_id": "full-agent-stream"}}

print("=== 스트리밍 실행 ===")
print("=" * 50)

for chunk in full_agent.stream(
    {"messages": [{"role": "user", "content": "LangChain이 무엇인지 검색해 주세요."}]},
    stream_mode="updates",                             # 노드별 업데이트
    config={**config_stream},
):
    for node_name, update in chunk.items():             # 노드별 반복
        print(f"\n--- {node_name} ---")
        if update and "messages" in update:
            for msg in update["messages"]:
                content = getattr(msg, "content", None) or ""
                if content:
                    print(f"  {content[:300]}")

=== 스트리밍 실행 ===
  [before_model] 메시지 1개 전달

--- log_model_input.before_model ---

--- model ---
  [after_model] 도구 호출: ['search_info']

--- log_model_output.after_model ---

--- tools ---
  LangChain은 LLM 기반 에이전트를 구축하기 위한 오픈소스 프레임워크입니다.
  [before_model] 메시지 3개 전달

--- log_model_input.before_model ---

--- model ---
  LangChain은 LLM(대형 언어 모델) 기반 에이전트를 구축하기 위한 오픈소스 프레임워크입니다. 즉, 챗봇이나 AI 어시스턴트와 같은 언어 모델을 활용한 다양한 애플리케이션을 쉽게 만들 수 있도록 도와주는 개발 도구입니다.
  [after_model] 응답: LangChain은 LLM(대형 언어 모델) 기반 에이전트를 구축하기 위한 오픈소스 프레임워크입니다. 즉, 챗봇이나 AI 어시스턴트와 같은 언어...

--- log_model_output.after_model ---


---

## 정리

| 항목 | 내용 |
|------|------|
| **다룬 기술** | LangChain v1, ChatOpenAI, LangGraph, python-dotenv |
| **핵심 개념** | 3-Layer 아키텍처(Model/Tools/Middleware), ReAct 패턴, invoke/stream/batch 호출, Messages 타입, @tool 데코레이터, InMemorySaver 메모리, 미들웨어 훅 |
